# The Import/Export AST Walker Notebook

The following source code graphs the dependencies, that is, the imports and exports of this application, https://mooreolith.github.io/graph-app/, but could in principle be adapted and applied to any javascript source since version 2015.

## Settings

Below, we clear the graph, set the default vertex size to 0.5, set arrows to true, and decrease attraction by a factor of 10 while increasing repulsion by a factor of ten. Decreasing attraction and increasing repulsion can prevent the simulation from blowing up (vertices rapidly flying out of bounds), and can be undone once the vertices and edges have been fully drawn, to increase the speed with which the simulation becomes stable (and discernible). Finally, we display only program labels, to get a general overview. 

In [0]:
graph.clear();
graph.defaults.vertex.size = .5;
graph.defaults.edge.arrow = true;
graph.constants.K = 2.0e-4;
graph.constants.f0 = 1.0e2;

labels = false;
programLabels = true;
specifierLabels = false;

## Source Map

Since imported packages can be referred to either by absolute path or patckage name, a sourcemap helps translate package names to full entry paths.

In [0]:
// sourceMap for imports
sourceMap = {
  "@codemirror/commands": "/node_modules/@codemirror/commands/dist/index.js",
  "@codemirror/lang-javascript": "/node_modules/@codemirror/lang-javascript/dist/index.js",
  "@codemirror/lang-markdown": "/node_modules/@codemirror/lang-markdown/dist/index.js",
  "@codemirror/state": "/node_modules/@codemirror/state/dist/index.js",
  "@codemirror/view": "/node_modules/@codemirror/view/dist/index.js",
  "codemirror": "/node_modules/codemirror/dist/index.js",
  "marked": "/node_modules/marked/lib/marked.esm.js",
  "three": "/node_modules/three/build/three.module.js",
  "OrbitControls": "/node_modules/three/examples/jsm/controls/OrbitControls.js",
  "SelectionBox": "/node_modules/three/examples/jsm/interactive/SelectionBox.js",
  "SelectionHelper": "/node_modules/three/examples/jsm/interactive/SelectionHelper.js",
  "@codemirror/autocomplete": "/node_modules/@codemirror/autocomplete/dist/index.js",
  "@codemirror/lang-css": "/node_modules/@codemirror/lang-css/dist/index.js",
  "@codemirror/lang-html": "/node_modules/@codemirror/lang-html/dist/index.js",
  "@codemirror/language": "/node_modules/@codemirror/language/dist/index.js",
  "@codemirror/lint": "/node_modules/@codemirror/lint/dist/index.js",
  "@codemirror/search": "/node_modules/@codemirror/search/dist/index.js",
  "@lezer/common": "/node_modules/@lezer/common/dist/index.js",
  "@lezer/css": "/node_modules/@lezer/css/dist/index.js",
  "@lezer/highlight": "/node_modules/@lezer/highlight/dist/index.js",
  "@lezer/html": "/node_modules/@lezer/html/dist/index.js",
  "@lezer/javascript": "/node_modules/@lezer/javascript/dist/index.js",
  "@lezer/lr": "/node_modules/@lezer/lr/dist/index.js",
  "@lezer/markdown": "/node_modules/@lezer/markdown/dist/index.js",
  "@marijn/find-cluster-break": "/node_modules/@marijn/find-cluster-break/src/index.js",
  "crelt": "/node_modules/crelt/index.js",
  "style-mod": "/node_modules/style-mod/src/style-mod.js",
  "w3c-keyname": "/node_modules/w3c-keyname/index.js"
};

## Helper Functions

* `await wait(10)` lets the invoking code sleep for ten milliseconds.
* programLabel(url) creates a label object for the programVertex function
* specifierLabel(text) creates a label object for the specifierVertex function.
* programVertex(url) draws a red, slightly larger vertex representing a module (program).
* importVertex() draws a yellow vertex representing the import, and subsequent vertices for the import specifiers (names).
* exportVertex() draws an orange vertex representing the export, and subsequent vertices for the export specifiers.
* specifierVertex(name, iore) draws either yellow or orange vertices, depending on if iore is `'i'` or `'e'`.

In [0]:
wait = async (ms) => new Promise(resolve => setTimeout(resolve, ms));

// draw a label for a program vertex
programLabel = (url) => {
  let text = url.endsWith('/dist/index.js') ? 
    url.split('/').at(-3) :
    url.split('/').at(-1);
  
  return programLabels ? {
    text,
    color: "black",
    fontSize: "16px",
    backgroundColor: "rgba(0,0,0,0.0)",
    border: "none"
  } : undefined;
}

// draw a label for a specifier vertex
specifierLabel = (text) => {
  return specifierLabels ? {
    text,
    color: "orange",
    fontSize: "12px",
    backgroundColor: "gray",
    border: "none"
  } : undefined;
}

// draw a program vertex
programVertex = (url) =>{
  return graph.addVertex({
    label: programLabel(url),
    color: "red",
    size: 3.0
  }).id;
}

// draw an import vertex
importVertex = () => {
  return graph.addVertex({
    color: "yellow"
  }).id;
}

// draw an export vertex
exportVertex = () => {
  return graph.addVertex({
    color: "orange"
  }).id;
}

// draw a specifier vertex
specifierVertex = (name, iore='i') => {
  return graph.addVertex({
    label: specifierLabel(name),
    color: iore == 'i' ? "yellow" : "orange"
  }).id;
}


## The follow Function

The following `follow` function calls an acorn walker on an acorn ast, implementing functions for `Program`, `ImportDeclaration`, and `ExportNamedDeclaration` nodes, to construct a dependency graph where each red cube represents a module, yellow cubes imports, and orange cubes stand in for exports.

In [0]:
const visited = new Set();
const imports = new Map();
const exports = new Map();
const follow = async (url) => {
  // recursive test, have we visited this node?
  if(visited.has(url)) return;
  // recursive advance, url is visited
  visited.add(url);

  // fetch url
  const response = await fetch(url);

  // read file
  const source = await response.text();
  
  // parse file to ast
  const ast = parser.parse(source);

  // perform a tree walk
  walker.ancestor(source, {
    Program(node, _, __){
      node.id ??= programVertex(url);
    },

    ImportDeclaration(node, _, ancestors){
      node.id ??= importVertex();

      const parent = ancestors.at(-2);
      parent.id ??= programVertex(url);
      graph.addEdge(node.id, parent.id);
      
      let filename = node.source.value;
      if(!filename.endsWith('.js') && !filename.endsWith('.mjs')){
        filename = sourceMap[filename];
      }
      
      for(const spec of node.specifiers){
        spec.id ??= specifierVertex(spec.imported.name, 'i');
        graph.addEdge(spec.id, node.id);
        imports.set(`${filename}-${spec.imported.name}`, spec.id);
        // console.log(`${filename}-${spec.imported.name}`)
      }

      follow(filename);
    },

    ExportNamedDeclaration(node, _, ancestors){
      node.id ??= exportVertex()
      
      const parent = ancestors.at(-2);
      parent.id ??= programVertex(url);
      graph.addEdge(parent.id, node.id);

      let filename = url;
      if(!filename.endsWith('.js') && !filename.endsWith('.mjs')){
        filename = sourceMap[filename];
      }

      for(const spec of node.specifiers){
        spec.id ??= specifierVertex(spec.exported.name, 'e');
        graph.addEdge(node.id, spec.id);
        graph.addEdge(spec.id, imports.get(`${filename}-${spec.exported.name}`));
      }
    }
  });
};

await follow('./graph-app.mjs');